<a href="https://colab.research.google.com/github/eeeewyz/NLP/blob/main/9_maskfilling_ner_text_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#一：mask_filtering task
from transformers import pipeline

unmasker = pipeline("fill-mask")


unmasker("This course will <mask> you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'score': 0.8891224265098572,
   'token': 6396,
   'token_str': ' teach',
   'sequence': '<s>This course will teach you all about<mask> models.</s>'},
  {'score': 0.08693923056125641,
   'token': 1137,
   'token_str': ' tell',
   'sequence': '<s>This course will tell you all about<mask> models.</s>'}],
 [{'score': 0.05115431174635887,
   'token': 8568,
   'token_str': ' automotive',
   'sequence': '<s>This course will<mask> you all about automotive models.</s>'},
  {'score': 0.017702210694551468,
   'token': 745,
   'token_str': ' building',
   'sequence': '<s>This course will<mask> you all about building models.</s>'}]]

In [7]:
#二：ner任务
from transformers import pipeline

# 显式指定默认模型以消除警告
#per代表person；org代表：organization；loc代表location
ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")
result = ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")
print(result)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER', 'score': np.float32(0.9981694), 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': np.float32(0.9796019), 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': np.float32(0.9932106), 'word': 'Brooklyn', 'start': 49, 'end': 57}]


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

#text_generation task
# 1. 加载 QA 专用模型
model_id = "distilbert/distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)
model.eval()

# 2. 准备输入
context = "My name is Sylvain and I work at Hugging Face in Brooklyn."
question = "am i man or femal"

# 3. Tokenizer 预处理
inputs = tokenizer(
    question,
    context,
    return_tensors="pt"
)

# 4. 模型推理
with torch.no_grad():
    outputs = model(**inputs)

# 5. 获取答案的起始和结束位置
start = outputs.start_logits.argmax(dim=-1).item()
end = outputs.end_logits.argmax(dim=-1).item()

# 6. 解码答案
answer = tokenizer.decode(
    inputs["input_ids"][0][start:end+1],
    skip_special_tokens=True
)

print("Answer:", answer)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Answer: Sylvain
